In [1]:
# Import data DataFrame from data_preprocessing.ipynb via %run magic
%run ./data_preprocessing.ipynb

int64
int64
Number of (store_id, item_id) groups with entirely missing sell_price: 0
Remaining nulls in sell_price: 0
Number of (store_id, item_id) groups with entirely missing sell_price: 0
Number of rows in data: 5,832,737
Cached 5,747,365 rows to /Users/deepakjacob/projects/Inventory-Forecasting-and-Modeling/data/collections/train_data.csv
<class 'pandas.core.frame.DataFrame'>
Index: 5747365 entries, 28 to 5832736
Data columns (total 29 columns):
 #   Column           Dtype         
---  ------           -----         
 0   id               object        
 1   item_id          object        
 2   dept_id          object        
 3   cat_id           object        
 4   store_id         object        
 5   state_id         object        
 6   d                object        
 7   sales            int16         
 8   date             datetime64[ns]
 9   wm_yr_wk         int64         
 10  weekday          object        
 11  wday             int64         
 12  month            int64 

In [8]:
# Encode categorical variables (category dtype for LightGBM)
cat_cols = ['item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'event_name_1']

for col in cat_cols:
    data[col] = data[col].astype('category')

### Train / Validation / Test split
- **Train:** Up to 2016-02-28 — fit model
- **Validation:** 2016-03-01 to 2016-03-28 — tune (early stopping, hyperparameters)
- **Test:** Final 28 days — evaluate final model only once

In [15]:
# Train / Validation / Test split by date
# Train: up to 2016-02-28  |  Validation: 2016-03-01 to 2016-03-28 (tune here)  |  Test: final 28 days (evaluate here)
drop_cols = ['sales', 'date', 'id', 'd']

train = data[data['date'] <= '2016-02-28']
valid = data[(data['date'] >= '2016-03-01') & (data['date'] <= '2016-03-28')]
test_end = data['date'].max()
test_start = test_end - pd.Timedelta(days=27)
test = data[(data['date'] >= test_start) & (data['date'] <= test_end)]

X_train = train.drop(columns=drop_cols)
y_train = train['sales']
X_valid = valid.drop(columns=drop_cols)
y_valid = valid['sales']
X_test = test.drop(columns=drop_cols)
y_test = test['sales']

print(f"Train:      {train['date'].min().date()} to {train['date'].max().date()}  ({len(train):,} rows)")
print(f"Validation: {valid['date'].min().date()} to {valid['date'].max().date()}  ({len(valid):,} rows) — tune here")
print(f"Test:       {test['date'].min().date()} to {test['date'].max().date()}  ({len(test):,} rows) — final evaluation")

Train:      2011-02-26 to 2016-02-28  (5,576,621 rows)
Validation: 2016-03-01 to 2016-03-28  (85,372 rows) — tune here
Test:       2016-03-28 to 2016-04-24  (85,372 rows) — final evaluation


In [16]:
categorical_features = [
    'item_id',
    'dept_id',
    'cat_id',
    'store_id',
    'state_id',
    'event_name_1',
    'event_type_1',
    'event_name_2',
    'event_type_2',
    'weekday'
]

for col in categorical_features:
    X_train[col] = X_train[col].astype('category')
    X_valid[col] = X_valid[col].astype('category')
    X_test[col] = X_test[col].astype('category')

In [17]:
import lightgbm as lgb

In [18]:
!pip install lightgbm
import lightgbm as lgb

model = lgb.LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.1,
    max_depth=8,
    num_leaves=128,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_valid, y_valid)],
    eval_metric='rmse',
    categorical_feature=categorical_features,
    callbacks=[lgb.early_stopping(50, verbose=True), lgb.log_evaluation(50)],
# Make sure LightGBM is installed before running this cell.
# If it is not installed, uncomment the line below:
# !pip install lightgbm

)

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.133462 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3816
[LightGBM] [Info] Number of data points in the train set: 5576621, number of used features: 25
[LightGBM] [Info] Start training from score 1.080844
Training until validation scores don't improve for 50 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[50]	valid_0's rmse: 1.7023	valid_0's l2: 2.89782
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

,boosting_type,'gbdt'
,num_leaves,128
,max_depth,8
,learning_rate,0.1
,n_estimators,1000
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [19]:
# Evaluate final model on TEST set (model was tuned on validation)
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

preds_test = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, preds_test))
print("Test RMSE:", rmse)

mae = mean_absolute_error(y_test, preds_test)
print("Test MAE:", mae)

r2 = r2_score(y_test, preds_test)
print("Test R² (accuracy):", r2)

Test RMSE: 1.6327436027665811
Test MAE: 0.8349054537905763
Test R² (accuracy): 0.7717351013956121


In [14]:
preds

array([1.22666535, 0.93571025, 0.93089792, ..., 0.0043878 , 0.00640871,
       0.00640871], shape=(85372,))

In [ ]:
data.head()

,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,...,snap_TX,snap_WI,sell_price,lag_7,lag_14,lag_28,rolling_mean_7,rolling_mean_14,rolling_mean_28,rolling_std_7
28,FOODS_1_005_CA_1_validation,FOODS_1_005,FOODS_1,FOODS,CA_1,CA,d_29,3,2011-02-26,11105,...,0,0,2.94,1.0,6.0,3.0,2.285714,2.714286,2.892857,1.380131
29,FOODS_1_005_CA_1_validation,FOODS_1_005,FOODS_1,FOODS,CA_1,CA,d_30,1,2011-02-27,11105,...,0,0,2.94,2.0,15.0,9.0,2.142857,1.714286,2.607143,1.463850
30,FOODS_1_005_CA_1_validation,FOODS_1_005,FOODS_1,FOODS,CA_1,CA,d_31,4,2011-02-28,11105,...,0,0,2.94,3.0,5.0,3.0,2.285714,1.642857,2.642857,1.603567
31,FOODS_1_005_CA_1_validation,FOODS_1_005,FOODS_1,FOODS,CA_1,CA,d_32,0,2011-03-01,11105,...,1,0,2.94,0.0,0.0,3.0,2.285714,1.642857,2.535714,1.603567
32,FOODS_1_005_CA_1_validation,FOODS_1_005,FOODS_1,FOODS,CA_1,CA,d_33,2,2011-03-02,11105,...,0,1,2.94,3.0,0.0,0.0,2.142857,1.785714,2.607143,1.573592
